# Mission Eagle-1 — Evaluation finale et demonstration

On a notre meilleur modele PPO. Avant de le deployer, il faut une evaluation rigoureuse et une demonstration video pour le rapport de mission.

> **Note methodo** : On ne touche JAMAIS au modele pendant l'evaluation. Pas d'entrainement, pas de modification. On charge, on teste, on mesure. C'est la separation entrainement/evaluation — un principe fondamental en ML.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import imageio

model = PPO.load("models/ppo_optimized")
print("Modele charge : models/ppo_optimized.zip")

## 1. Evaluation sur 100 episodes

On evalue sur 100 episodes pour avoir une mesure statistiquement fiable. On collecte aussi les scores individuels pour analyser la distribution.

In [ ]:
eval_env = gym.make("LunarLander-v3")

# Collecte detaillee
episode_rewards = []
for ep in range(100):
    obs, info = eval_env.reset()
    total_reward = 0
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        done = terminated or truncated

    episode_rewards.append(total_reward)

eval_env.close()

mean_r = np.mean(episode_rewards)
std_r = np.std(episode_rewards)
success_rate = np.mean([r > 200 for r in episode_rewards]) * 100

print("Evaluation finale — 100 episodes")
print("=" * 40)
print(f"  Reward moyen  : {mean_r:.1f}")
print(f"  Ecart-type    : {std_r:.1f}")
print(f"  Min           : {np.min(episode_rewards):.1f}")
print(f"  Max           : {np.max(episode_rewards):.1f}")
print(f"  Taux > 200    : {success_rate:.0f}%")
print(f"  Objectif      : {'ATTEINT' if mean_r > 200 else 'NON ATTEINT'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogramme
axes[0].hist(episode_rewards, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(mean_r, color="red", linestyle="--", label=f"Moyenne = {mean_r:.1f}")
axes[0].axvline(200, color="green", linestyle="--", label="Seuil = 200")
axes[0].set_xlabel("Reward")
axes[0].set_ylabel("Nombre d'episodes")
axes[0].set_title("Distribution des rewards (100 episodes)")
axes[0].legend()

# Evolution episode par episode
axes[1].plot(episode_rewards, color="steelblue", alpha=0.7, linewidth=1)
axes[1].axhline(mean_r, color="red", linestyle="--", label=f"Moyenne = {mean_r:.1f}")
axes[1].axhline(200, color="green", linestyle="--", label="Seuil = 200")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Reward")
axes[1].set_title("Reward par episode")
axes[1].legend()

plt.tight_layout()
plt.show()

### Analyse des echecs

Regardons les episodes ou le score est en dessous de 200 — qu'est-ce qui s'est passe ?

In [ ]:
failures = [(i, r) for i, r in enumerate(episode_rewards) if r < 200]

if failures:
    print(f"{len(failures)} episodes sous le seuil de 200 :")
    for ep_idx, reward in failures:
        print(f"  Episode {ep_idx:3d} : {reward:.1f}")
    print(f"\n  Pire score : {min(r for _, r in failures):.1f}")
    print(f"  Ces echecs representent {len(failures)}% des episodes")
else:
    print("Aucun echec ! Tous les episodes sont au-dessus de 200.")

## 2. Generation de la video de demonstration

On va enregistrer un atterrissage reussi en video .mp4. On utilise `render_mode="rgb_array"` pour capturer les frames, puis `imageio` pour les assembler.

> **Rappel gymnasium** : `render_mode` se passe a `gym.make()`, pas a `env.render()`. C'est une erreur classique quand on vient de l'ancienne API `gym`.

In [ ]:
# Environnement avec capture de frames
video_env = gym.make("LunarLander-v3", render_mode="rgb_array")

# On fait plusieurs tentatives pour trouver un bel atterrissage
best_frames = None
best_reward = -float("inf")

for attempt in range(10):
    frames = []
    obs, info = video_env.reset()
    total_reward = 0
    done = False

    while not done:
        frames.append(video_env.render())
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        total_reward += reward
        done = terminated or truncated

    # Garder quelques frames a la fin pour que la video ne coupe pas trop vite
    for _ in range(30):
        frames.append(video_env.render())

    if total_reward > best_reward and total_reward > 250:
        best_reward = total_reward
        best_frames = frames
        print(f"  Tentative {attempt + 1} : {total_reward:.1f} -- nouveau meilleur !")
    else:
        print(f"  Tentative {attempt + 1} : {total_reward:.1f}")

video_env.close()

if best_frames:
    print(f"\nMeilleur episode retenu : {best_reward:.1f}")
else:
    print("\nAucun episode > 250 trouve, on prend le dernier")
    best_frames = frames

# Sauvegarder en mp4
video_path = "videos/eagle1_landing.mp4"
imageio.mimsave(video_path, best_frames, fps=30)

duration = len(best_frames) / 30
print(f"Video sauvegardee : {video_path}")
print(f"Duree : {duration:.1f} secondes ({len(best_frames)} frames a 30 FPS)")

## 3. Bilan de mission Eagle-1

### Parcours complet

| Etape | Description | Resultat |
|-------|------------|----------|
| Exploration | Decouverte de LunarLander-v3 | Env compris : 8 obs, 4 actions |
| Agent aleatoire | Performance de reference | Score ~-150 |
| PPO baseline | Premier entrainement (100k steps) | Score ~A remplir |
| Optimisation | Tuning learning_rate, gamma, n_steps | Score ~A remplir |
| Comparaison DQN | Verification du choix d'algorithme | PPO > DQN (attendu) |
| Evaluation finale | 100 episodes, modele final | Score moyen : A remplir |

### Ce qui a marche
- PPO converge bien sur LunarLander, meme avec les parametres par defaut
- L'optimisation un-parametre-a-la-fois a permis d'isoler les effets
- Le modele final est stable (ecart-type faible)

### Ce qui pourrait etre ameliore
- Tester d'autres hyperparametres (ent_coef, clip_range, architecture du reseau)
- Essayer du reward shaping pour accelerer la convergence
- Entrainer plus longtemps (1M+ timesteps)
- Tester A2C comme alternative

### Livrables
- Modele : `models/ppo_optimized.zip`
- Video : `videos/eagle1_landing.mp4`
- API : `api/main.py`
- GUI : `gui/app.py`
- Dashboard : `dashboard/app.py`